In [1]:
import pandas as pd

# 1) 读取数据
df = pd.read_csv("final_merged_data.csv")

# 2) 时间列处理：转换为 datetime，并提取 hour 与 day_of_week
df["last_reported"] = pd.to_datetime(df["last_reported"], errors="coerce")
df["hour"] = df["last_reported"].dt.hour
df["day_of_week"] = df["last_reported"].dt.dayofweek

# 3) 目标列（回归问题）
target_col = "num_bikes_available"

# 4) 特征选择：仅保留指定特征 + 目标列
feature_cols = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "max_air_temperature_celsius",
    "max_relative_humidity_percent",
    "max_barometric_pressure_hpa",
]
keep_cols = feature_cols + [target_col]

df_selected = df[keep_cols].copy()

# 5) 删除保留列中含有缺失值的行
df_clean = df_selected.dropna()

# 6) 打印清洗后数据概览
print("Cleaned data shape:", df_clean.shape)
print("\nFirst 5 rows:")
print(df_clean.head())

Cleaned data shape: (298946, 8)

First 5 rows:
   station_id  hour  day_of_week  month  max_air_temperature_celsius  \
0          10     0            6     12                        14.01   
1         100     0            6     12                        14.01   
2         109     0            6     12                        14.01   
3          11     0            6     12                        14.01   
4         114     0            6     12                        14.01   

   max_relative_humidity_percent  max_barometric_pressure_hpa  \
0                           84.3                      1002.56   
1                           84.3                      1002.56   
2                           84.3                      1002.56   
3                           84.3                      1002.56   
4                           84.3                      1002.56   

   num_bikes_available  
0                   15  
1                   17  
2                   20  
3                    1  
4   

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1) 划分特征与目标
X = df_clean[feature_cols]
y = df_clean[target_col]

# 2) 按 80/20 切分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3) 实例化模型
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=50, random_state=42),
    "KNeighbors Regressor": KNeighborsRegressor(),
}

# 4) 训练并评估
results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "R2 Score": r2,
    })

# 5) 结果汇总展示
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="MAE", ascending=True).reset_index(drop=True)
results_df["MAE"] = results_df["MAE"].round(4)
results_df["R2 Score"] = results_df["R2 Score"].round(4)

print("Model Evaluation on Test Set (sorted by MAE):")
display(results_df)

Model Evaluation on Test Set (sorted by MAE):


,Model,MAE,R2 Score
0,Random Forest Regressor,0.9707,0.9649
1,Decision Tree Regressor,1.0101,0.9293
2,KNeighbors Regressor,4.0336,0.6414
3,Linear Regression,8.1363,-0.0000


In [3]:
import joblib

# 1) 自动识别 R2 Score 最高的模型
best_row = results_df.loc[results_df["R2 Score"].idxmax()]
best_model_name = best_row["Model"]
best_model = models[best_model_name]

# 2) 保存最佳模型到当前目录
joblib.dump(best_model, "best_bike_model.joblib")

print("最佳模型已成功保存")

最佳模型已成功保存
